<a href="https://colab.research.google.com/github/aditi19sahu/eRes/blob/main/eRes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Base-year calibration data
- **Road-fuel use by vehicle segment.** Road transport burns about half of India's ≈5.6 mb/d of oil. The split across two-wheelers, three-wheelers, cars, buses and trucks is the author's estimate, consistent with petrol and diesel consumption patterns.
- **Electricity needed per litre displaced.** An EV doesn't remove oil for free; it adds grid demand. The conversion follows from comparing fuel economy and electric efficiency. For example, a petrol two-wheeler does about 50 km/L, while an electric one uses about 25 Wh/km. So one litre of petrol displaced needs about 50 × 0.025 ≈ 1.3 kWh. A diesel truck at about 4 km/L against about 1.2 kWh/km gives about 4.8 kWh per litre.


In [ ]:
import copy

YEARS = list(range(2025, 2048))          # simulation horizon: 2025 (calibration year) to 2047

# Observed 2025 values (calendar year; Ember / Our World in Data unless noted)
BASE = dict(
    gen_TWh=2081.6,                       # total electricity generation, TWh
    coal=1474.15, gas=48.54, oil=4.10,    # fossil generation by fuel, TWh
    nuclear=53.83, hydro=177.97, solar=195.99, wind=103.98, bio=23.04,   # clean generation by source, TWh
    cap_solar=140.0, cap_wind=54.0,       # installed capacity at end-2025, GW (approx., MNRE/PIB)
    cap_nuc=8.8, cap_hydro=52.0,          # GW
    road_oil_mbd=2.70,                    # road-transport oil use, million barrels/day (≈ half of total)
    nonroad_oil_mbd=2.90,                 # all other oil use (LPG, industry, aviation, petrochemicals), mb/d
    dom_crude_mbd=0.55,                   # domestic crude production, mb/d (held flat)
    gdp_usd_T=4.0,                        # GDP, trillion US$ (real 2025)
    pop_bn=1.46,                          # population, billions
)

# Road-oil use by vehicle segment in 2025, mb/d (author's split; sums to 2.70)
SEG = dict(two_w=0.55, three_w=0.05, cars=0.55, buses=0.25, trucks=1.30)

# Grid electricity needed per litre of fuel an EV displaces (kWh/L) = (km per litre) × (kWh per km)
KWH_PER_L = dict(two_w=1.3, three_w=1.8, cars=2.3, buses=4.5, trucks=4.8)

Model parameters


1. **Capacity factors (CF).** Annual generation = capacity × CF × 8,760 hours.
   - Solar's 18.5% and wind's 23% are back-solved from 2025 data.
   - Wind's CF rises to 30% by 2047 as taller turbines reach steadier winds.
   - Nuclear's 80% reflects a well-run reactor fleet.
2. **Growth rates.** Electricity demand, road and non-road oil demand, and real GDP. All compound annually: $X_t = X_{t-1}(1+g)$.
3. **Emission and fuel factors.**
   - Coal emits about 0.98 tCO₂ per MWh at India's fleet efficiency; gas about 0.45.
   - Indian coal plants burn about 0.63 kg of coal per kWh (calibrated to 2025).
4. **Costs, in real 2025 ₹/kWh.**
   - *Clean sources* use a **levelised cost** (LCOE): lifetime cost ÷ lifetime output. It declines each year at a learning rate, down to a floor.
   - *Coal* is split into a **variable** (fuel) part and a **fixed** (capital) part. That split is the heart of the report's cost-structure argument (Lens A): fixed costs per kWh rise when plants run less.
   - *Storage* is priced as an **adder** per kWh time-shifted (Ember: ₹4.0–4.5).
   - *Nuclear* has separate costs for the existing and new fleet.
5. **Storage threshold.** The share of demand that solar can serve directly in daylight before its output must be stored (20%). Round-trip efficiency is 90%.
   - **The threshold is a calibration, not a physical law.** It was chosen so the current-policy scenario's 2032 storage need roughly matches CEA's National Electricity Plan (≈236 GWh of batteries plus ≈175 GWh of pumped hydro).

In [ ]:
PARAMS = dict(
    # 1. Capacity factors (fraction of the year at full output)
    cf_solar=0.185, cf_wind_2025=0.23, cf_wind_2047=0.30, cf_nuc=0.80, cf_hydro=0.38,
    bio_TWh=25, gas_TWh=45, oil_TWh=4,            # small sources held flat after 2025, TWh/yr

    # 2. Growth rates (fraction per year, compounding)
    demand_growth_2025_35=0.055, demand_growth_2035_47=0.045,   # underlying electricity demand (before EVs)
    road_oil_growth=0.035, nonroad_oil_growth=0.030,            # oil demand if nothing electrified
    gdp_usd_growth_2025_35=0.065, gdp_usd_growth_2035_47=0.055, # real GDP
    pop_2035=1.55, pop_2047=1.62,                               # population anchors, billions

    # 3. Emission and fuel factors
    co2_coal=0.98, co2_gas=0.45,       # tonnes CO2 per MWh generated
    coal_kg_per_kWh=0.63,              # kg of coal burned per kWh (calibrated to 2025)

    # 4. Costs, real 2025 INR per kWh
    lcoe_solar_2025=2.6, solar_decline=0.015, solar_floor=2.0,          # solar LCOE, annual decline, floor
    lcoe_wind_2025=3.4, wind_decline=0.01, wind_floor=2.8,              # wind LCOE, annual decline, floor
    storage_adder_2025=4.25, storage_decline=0.04, storage_floor=2.0,   # cost per kWh time-shifted
    coal_var=2.9,                      # coal variable (fuel + O&M) cost per kWh
    coal_fixed_at_69=1.9,              # coal fixed (capital) cost per kWh *when the fleet runs at 69% PLF*
    nuc_existing=3.8, nuc_new=6.0,     # nuclear: existing fleet vs new build (CEA roadmap)
    hydro=4.0, gas=8.0, bio=6.0,       # other sources, flat

    # 5. Storage rule of thumb
    direct_solar_absorb=0.20,          # share of demand solar can serve directly before shifting is needed
    rte=0.90,                          # battery round-trip efficiency
)